# 02 · 캘리브레이션과 좌표계 (Phase 2)

**목표**: `calib_velo_to_cam.txt`, `calib_cam_to_cam.txt` 를 읽고, Velodyne → 카메라 → 정류 카메라 → 픽셀로 이어지는
**변환 체인**을 손으로 따라가며 부호와 순서를 확인한다.

| 항목 | 내용 |
|---|---|
| 입력 | 캘리브레이션 텍스트 2개 (`R`, `T`, `R_rect_00`, `P_rect_02`, `S_rect_02`) |
| 출력 | `T_velo_to_cam (4×4)`, `R_rect_00 (4×4)`, `P_rect_02 (3×4)`, `P_velo_to_img (3×4)` |
| 좌표계 | Velodyne (x 전방, y 좌, z 상) → cam0 (x 우, y 아래, z 전방) → rect → image_02 픽셀 (u 우, v 아래) |
| 핵심 수식 | $y = P_{rect,02}\,R_{rect,00}\,T_{velo\to cam}\,x_{velo}$, $(u, v) = (y_0/y_2,\ y_1/y_2)$ |
| 실패 사례 | 곱하는 순서 반대로 · `R_rect_02` 를 잘못 사용 · 텍스트 반올림으로 R 이 정확히 직교가 아님 |
| 평가 | 직교성 $RR^T = I$, round-trip 오차 < 1e-6, 알려진 점의 부호, `tests/test_calibration.py` |

용어
- **강체 변환(rigid transform)**: 회전 + 이동만 있는 변환. 길이와 각도를 보존한다. $p' = Rp + t$.
- **동차 좌표(homogeneous)**: 3D 점 뒤에 1 을 붙인 $[x, y, z, 1]^T$. 회전과 이동을 4×4 행렬 곱 하나로 쓸 수 있다.
- **정류(rectification)**: 스테레오 카메라 두 대의 이미지 평면이 평행하도록 가상으로 회전시키는 것. KITTI 이미지(`image_0x/data`)는 이미 정류·왜곡보정된 상태다.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from perceptrack3d.config import load_config
from perceptrack3d.geometry.calibration import read_calib_file, KittiCalibration
from perceptrack3d.geometry.transforms import to_homogeneous, apply_transform, invert_rigid, make_rigid, nearest_rotation

np.set_printoptions(precision=6, suppress=True, linewidth=120)
cfg = load_config()
calib_dir = Path(cfg["dataset"]["calib_dir"])
OUT = Path(cfg["outputs"]["dir"]) / "phase2"
OUT.mkdir(parents=True, exist_ok=True)
print("calib_dir:", calib_dir)
print(sorted(p.name for p in calib_dir.glob("*.txt")))

calib_dir: /home/ingon/datasets/KITTI/raw/2011_09_26
['calib_cam_to_cam.txt', 'calib_imu_to_velo.txt', 'calib_velo_to_cam.txt']


## 1. 파일 원문

`key: 값 값 값 ...` 형식이다. 행렬은 **행 우선(row-major)** 으로 펼쳐져 있으므로 `reshape(3, 3)` / `reshape(3, 4)` 로 되돌린다.

In [ ]:
print("=== calib_velo_to_cam.txt ===")
print((calib_dir / "calib_velo_to_cam.txt").read_text())
print("=== calib_cam_to_cam.txt (image_02 관련 줄만) ===")
for line in (calib_dir / "calib_cam_to_cam.txt").read_text().splitlines():
    if line.split(":")[0] in ("S_02", "K_02", "D_02", "R_02", "T_02", "S_rect_02", "R_rect_02", "P_rect_02", "R_rect_00", "P_rect_00"):
        print(line)

=== calib_velo_to_cam.txt ===
calib_time: 15-Mar-2012 11:37:16
R: 7.533745e-03 -9.999714e-01 -6.166020e-04 1.480249e-02 7.280733e-04 -9.998902e-01 9.998621e-01 7.523790e-03 1.480755e-02
T: -4.069766e-03 -7.631618e-02 -2.717806e-01
delta_f: 0.000000e+00 0.000000e+00
delta_c: 0.000000e+00 0.000000e+00

=== calib_cam_to_cam.txt (image_02 관련 줄만) ===
R_rect_00: 9.999239e-01 9.837760e-03 -7.445048e-03 -9.869795e-03 9.999421e-01 -4.278459e-03 7.402527e-03 4.351614e-03 9.999631e-01
P_rect_00: 7.215377e+02 0.000000e+00 6.095593e+02 0.000000e+00 0.000000e+00 7.215377e+02 1.728540e+02 0.000000e+00 0.000000e+00 0.000000e+00 1.000000e+00 0.000000e+00
S_02: 1.392000e+03 5.120000e+02
K_02: 9.597910e+02 0.000000e+00 6.960217e+02 0.000000e+00 9.569251e+02 2.241806e+02 0.000000e+00 0.000000e+00 1.000000e+00
D_02: -3.691481e-01 1.968681e-01 1.353473e-03 5.677587e-04 -6.770705e-02
R_02: 9.999758e-01 -5.267463e-03 -4.552439e-03 5.251945e-03 9.999804e-01 -3.413835e-03 4.570332e-03 3.389843e-03 9.999838e-01


## 2. 각 행렬의 의미

### `calib_velo_to_cam.txt`
- `R` (3×3), `T` (3,): $p_{cam0} = R\,p_{velo} + T$. Velodyne 점을 **카메라 0(왼쪽 흑백, 기준 카메라)** 프레임으로 옮긴다.
  `T` 를 보면 Velodyne 원점은 cam0 기준 $(-0.004, -0.076, -0.272)$ m, 즉 카메라보다 **0.27 m 뒤(−z), 7.6 cm 위(−y)** 에 있다.

### `calib_cam_to_cam.txt` 에서 쓰는 것
- `R_rect_00` (3×3): cam0 를 정류 프레임으로 돌리는 회전. **image_02 에 투영할 때도 이것을 쓴다** (`R_rect_02` 가 아님 — devkit 규정).
- `P_rect_02` (3×4): 정류 프레임의 3D 점을 image_02 픽셀로 보내는 투영 행렬
  $$P_{rect,02} = \begin{bmatrix} f & 0 & c_x & t_x \\ 0 & f & c_y & t_y \\ 0 & 0 & 1 & t_z \end{bmatrix}$$
  $f$ = 초점거리(px), $(c_x, c_y)$ = 주점(광축이 지나는 픽셀). 네 번째 열 $t$ 는 카메라 2 가 cam0 에서 옆으로 떨어진 거리(**baseline**)를
  $f \cdot b$ 형태로 담은 것이다. $t_x = 44.86 = 721.5 \times 0.062$ → 카메라 2 는 cam0 의 **왼쪽 6.2 cm** 에 있다 (부호는 cam0 → cam2 이동을 뜻함).
- `S_rect_02` (2,): 정류 이미지 크기 **(width, height) = (1242, 375)**. numpy shape 은 `(375, 1242)` 로 **순서가 반대**임에 주의.

### 쓰지 않는 것
- `K_02`, `D_02`, `R_02`, `T_02`: 왜곡이 있는 **원본(비정류)** 이미지용. 우리는 정류된 `image_02/data` 를 쓰므로 필요 없다.

In [ ]:
vc = read_calib_file(calib_dir / "calib_velo_to_cam.txt")
cc = read_calib_file(calib_dir / "calib_cam_to_cam.txt")
R_raw = vc["R"].reshape(3, 3)
t_raw = vc["T"]
R_rect_raw = cc["R_rect_00"].reshape(3, 3)
P_rect_02 = cc["P_rect_02"].reshape(3, 4)
S_rect_02 = cc["S_rect_02"]
print("R (velo→cam0):\n", R_raw)
print("T (velo→cam0):", t_raw)
print("R_rect_00:\n", R_rect_raw)
print("P_rect_02:\n", P_rect_02)
print("S_rect_02 (width, height):", S_rect_02)
f, cx, cy, tx = P_rect_02[0, 0], P_rect_02[0, 2], P_rect_02[1, 2], P_rect_02[0, 3]
print(f"\nf = {f:.2f} px, (cx, cy) = ({cx:.2f}, {cy:.2f}) px, baseline = tx / f = {tx / f:+.4f} m")

R (velo→cam0):
 [[ 0.007534 -0.999971 -0.000617]
 [ 0.014802  0.000728 -0.99989 ]
 [ 0.999862  0.007524  0.014808]]
T (velo→cam0): [-0.00407  -0.076316 -0.271781]
R_rect_00:
 [[ 0.999924  0.009838 -0.007445]
 [-0.00987   0.999942 -0.004278]
 [ 0.007403  0.004352  0.999963]]
P_rect_02:
 [[721.5377     0.       609.5593    44.85728 ]
 [  0.       721.5377   172.854      0.216379]
 [  0.         0.         1.         0.002746]]
S_rect_02 (width, height): [1242.  375.]

f = 721.54 px, (cx, cy) = (609.56, 172.85) px, baseline = tx / f = +0.0622 m


## 3. 네 개의 좌표계

| 이름 | 원점 | 축 | 단위 |
|---|---|---|---|
| Velodyne (`velo`) | LiDAR 중심 | **x 전방, y 좌, z 상** | m |
| Camera 0 (`cam0`) | 왼쪽 흑백 카메라 광학 중심 | **x 우, y 아래, z 전방** | m |
| Rectified (`rect`) | cam0 와 같음 | cam0 를 `R_rect_00` 으로 살짝(≈1°) 돌린 것 | m |
| Image 02 (`img`) | 이미지 좌상단 | u 우, v 아래 | px |

Velodyne 과 카메라는 **축 이름이 다르게 배치**되어 있다. 대략적으로
$x_{velo} \to z_{cam}$ (전방), $y_{velo} \to -x_{cam}$ (좌 = 카메라의 −우), $z_{velo} \to -y_{cam}$ (상 = 카메라의 −아래).
`R` 을 보면 정확히 이 순열(부호 포함)에 작은 회전이 더해진 형태다. 아래 그림은 카메라 축을 Velodyne 프레임에 그린 것이다.

In [ ]:
T_velo_to_cam_raw = make_rigid(R_raw, t_raw)            # (4,4) 동차
T_cam_to_velo = invert_rigid(make_rigid(nearest_rotation(R_raw), t_raw))
cam_origin_in_velo = T_cam_to_velo[:3, 3]
cam_axes_in_velo = T_cam_to_velo[:3, :3]                # 열 = cam x, y, z 축을 velo 에서 본 방향
print("cam0 원점 (velo 프레임):", cam_origin_in_velo.round(3), "m  → 카메라는 LiDAR 보다 앞(+x), 아래(-z)")
for name, col in zip("xyz", cam_axes_in_velo.T):
    print(f"cam {name} 축 (velo 에서 본 방향): {col.round(3)}")

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")
L = 1.0
for vec, c, lab in zip(np.eye(3), ["tab:red", "tab:green", "tab:blue"], ["velo x (fwd)", "velo y (left)", "velo z (up)"]):
    ax.quiver(0, 0, 0, *(L * vec), color=c, linewidth=2.5, arrow_length_ratio=0.15, label=lab)
o = cam_origin_in_velo
for vec, c, lab in zip(cam_axes_in_velo.T, ["tab:red", "tab:green", "tab:blue"], ["cam x (right)", "cam y (down)", "cam z (fwd)"]):
    ax.quiver(*o, *(0.6 * L * vec), color=c, linestyle="dashed", linewidth=1.5, arrow_length_ratio=0.2, label=lab)
ax.scatter(*o, color="k", s=30); ax.text(*o, "  cam0", fontsize=9)
ax.text(0, 0, 0.05, "velo", fontsize=9)
ax.set_xlim(-0.5, 1.2); ax.set_ylim(-1.0, 1.0); ax.set_zlim(-0.8, 1.0)
ax.set_xlabel("velo x"); ax.set_ylabel("velo y"); ax.set_zlabel("velo z")
ax.set_box_aspect((1.7, 2.0, 1.8)); ax.view_init(elev=22, azim=-135)
ax.legend(loc="upper left", fontsize=8); ax.set_title("Sensor frames drawn in the Velodyne frame (solid = velo, dashed = cam0)")
fig.tight_layout(); fig.savefig(OUT / "frames_axes.png", dpi=130); plt.show()

cam0 원점 (velo 프레임): [ 0.273 -0.002 -0.072] m  → 카메라는 LiDAR 보다 앞(+x), 아래(-z)
cam x 축 (velo 에서 본 방향): [ 0.008 -1.    -0.001]
cam y 축 (velo 에서 본 방향): [ 0.015  0.001 -1.   ]
cam z 축 (velo 에서 본 방향): [1.    0.008 0.015]


/tmp/ipykernel_192089/3172337578.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUT / "frames_axes.png", dpi=130); plt.show()


## 4. 4×4 동차 강체 변환

$p' = Rp + t$ 를 곱셈 하나로 쓰기 위해 점에 1 을 붙인다:
$$\begin{bmatrix} p' \\ 1 \end{bmatrix} = \underbrace{\begin{bmatrix} R & t \\ 0\;0\;0 & 1 \end{bmatrix}}_{T\ (4\times4)} \begin{bmatrix} p \\ 1 \end{bmatrix}$$

역변환은 `np.linalg.inv` 없이 회전의 직교성 $R^{-1} = R^T$ 로 얻는다:
$$T^{-1} = \begin{bmatrix} R^T & -R^T t \\ 0 & 1 \end{bmatrix}$$

**함정**: 텍스트 파일의 회전은 유효숫자 7자리로 반올림되어 있어 $RR^T \ne I$ (오차 ~1e-7) 이다. 그대로 $R^T$ 를 역으로 쓰면
먼 점(70 m)에서 round-trip 오차가 ~2e-6 m 가 되어 "정확히 되돌아온다" 는 성질이 깨진다.
그래서 `KittiCalibration.from_dir` 는 SVD 로 **가장 가까운 회전 행렬**로 보정한다 (`nearest_rotation`). 값 변화는 1e-7 수준이라 투영에는 영향이 없다.

In [ ]:
I = np.eye(3)
print("raw R:        |R Rᵀ − I|max =", np.abs(R_raw @ R_raw.T - I).max(), " det =", np.linalg.det(R_raw))
print("raw R_rect_00:|R Rᵀ − I|max =", np.abs(R_rect_raw @ R_rect_raw.T - I).max(), " det =", np.linalg.det(R_rect_raw))
R_fix = nearest_rotation(R_raw)
print("보정 후 R:    |R Rᵀ − I|max =", np.abs(R_fix @ R_fix.T - I).max(), " |R_fix − R_raw|max =", np.abs(R_fix - R_raw).max())

# round-trip 오차 비교 (velo → rect → velo), 먼 점 포함
pts = np.array([[70.0, 40.0, 3.0], [10.0, 0.0, 0.0], [0.0, 0.0, 0.0], [-30.0, -20.0, -2.0]])
for label, R_use, Rr_use in (("raw", R_raw, R_rect_raw), ("orthonormalized", R_fix, nearest_rotation(R_rect_raw))):
    T = make_rigid(Rr_use, np.zeros(3)) @ make_rigid(R_use, t_raw)
    back = apply_transform(invert_rigid(T), apply_transform(T, pts))
    print(f"{label:16s} round-trip max error = {np.abs(back - pts).max():.2e} m")

raw R:        |R Rᵀ − I|max = 8.996937661542859e-08  det = 1.0000000420780264
raw R_rect_00:|R Rᵀ − I|max = 7.858283346262596e-08  det = 0.9999999363909579
보정 후 R:    |R Rᵀ − I|max = 8.881784197001252e-16  |R_fix − R_raw|max = 4.5000244530690736e-08
raw              round-trip max error = 1.85e-06 m
orthonormalized  round-trip max error = 6.39e-14 m


In [ ]:
calib = KittiCalibration.from_dir(calib_dir)
print("T_velo_to_cam (4x4):\n", calib.T_velo_to_cam)
print("R_rect_00 (4x4, 회전만 → 마지막 열 0):\n", calib.R_rect_00)
print("P_rect_02 (3x4):\n", calib.P_rect_02)
print("image_size (W, H):", calib.image_size, " image_shape (H, W):", calib.image_shape)

T_velo_to_cam (4x4):
 [[ 0.007534 -0.999971 -0.000617 -0.00407 ]
 [ 0.014802  0.000728 -0.99989  -0.076316]
 [ 0.999862  0.007524  0.014808 -0.271781]
 [ 0.        0.        0.        1.      ]]
R_rect_00 (4x4, 회전만 → 마지막 열 0):
 [[ 0.999924  0.009838 -0.007445  0.      ]
 [-0.00987   0.999942 -0.004278  0.      ]
 [ 0.007403  0.004352  0.999963  0.      ]
 [ 0.        0.        0.        1.      ]]
P_rect_02 (3x4):
 [[721.5377     0.       609.5593    44.85728 ]
 [  0.       721.5377   172.854      0.216379]
 [  0.         0.         1.         0.002746]]
image_size (W, H): (1242, 375)  image_shape (H, W): (375, 1242)


## 5. 변환 체인 (KITTI raw devkit `readme.txt` 의 식)

$$
\underbrace{\begin{bmatrix} u\,d \\ v\,d \\ d \end{bmatrix}}_{y\ (3\times1)}
= \underbrace{P_{rect,02}}_{3\times4}\;
  \underbrace{R_{rect,00}}_{4\times4}\;
  \underbrace{T_{velo\to cam}}_{4\times4}\;
  \underbrace{\begin{bmatrix} x \\ y \\ z \\ 1 \end{bmatrix}}_{x_{velo}\ (4\times1)},
\qquad (u, v) = \left(\frac{y_0}{y_2}, \frac{y_1}{y_2}\right)
$$

- 행렬 곱은 **오른쪽부터** 적용된다: 먼저 velo → cam0, 그 다음 정류 회전, 마지막으로 투영.
- $d = y_2$ 는 카메라 앞쪽 깊이(m). $d \le 0$ 이면 카메라 뒤에 있는 점이라 나누면 안 된다 (Phase 3 에서 제거).
- 세 행렬을 미리 곱해 둔 것이 `P_velo_to_img` (3×4). 점이 12만 개라도 행렬 곱 한 번이면 끝난다.

In [ ]:
print("T_velo_to_rect = R_rect_00 @ T_velo_to_cam:\n", calib.T_velo_to_rect)
print("\nP_velo_to_img = P_rect_02 @ R_rect_00 @ T_velo_to_cam  (3x4):\n", calib.P_velo_to_img)

T_velo_to_rect = R_rect_00 @ T_velo_to_cam:
 [[ 0.000235 -0.999944 -0.010563 -0.002797]
 [ 0.010449  0.010565 -0.99989  -0.075109]
 [ 0.999945  0.000124  0.010451 -0.272133]
 [ 0.        0.        0.        1.      ]]

P_velo_to_img = P_rect_02 @ R_rect_00 @ T_velo_to_cam  (3x4):
 [[ 609.695401 -721.421614   -1.251258 -123.041811]
 [ 180.384199    7.644798 -719.651482 -101.016692]
 [   0.999945    0.000124    0.010451   -0.269387]]


## 6. 손으로 점 몇 개 변환해 보기 — 부호 확인

| Velodyne 점 | 기대하는 rect 결과 | 이유 |
|---|---|---|
| (0, 0, 0) 원점 | z ≈ −0.27 (카메라 뒤) | LiDAR 는 카메라보다 뒤에 있다 |
| (10, 0, 0) 전방 10 m | z ≈ 9.73, x ≈ 0, y ≈ 0 | 전방 = 카메라 +z, 원점 차이 0.27 m 만큼 가까워짐 |
| (10, 5, 0) 왼쪽 5 m | x ≈ −5 | Velodyne +y(좌) = 카메라 −x |
| (10, 0, 3) 위 3 m | y ≈ −3 | Velodyne +z(상) = 카메라 −y |

In [ ]:
test_pts = np.array([[0.0, 0.0, 0.0], [10.0, 0.0, 0.0], [10.0, 5.0, 0.0], [10.0, 0.0, 3.0]])
labels = ["velo origin", "10 m forward", "10 m fwd, 5 m left", "10 m fwd, 3 m up"]
cam0 = apply_transform(calib.T_velo_to_cam, test_pts)
rect = calib.velo_to_rect(test_pts)
print(f"{'point':20s} {'velo (x,y,z)':>22s} | {'cam0 (x,y,z)':>26s} | {'rect (x,y,z)':>26s}")
for lab, p, c, r in zip(labels, test_pts, cam0, rect):
    print(f"{lab:20s} {str(p.round(2)):>22s} | {str(c.round(3)):>26s} | {str(r.round(3)):>26s}")
print("\n확인: 전방→rect z 양수?", rect[1, 2] > 9, "| 좌→rect x 음수?", rect[2, 0] < -4, "| 상→rect y 음수?", rect[3, 1] < -2)

point                          velo (x,y,z) |               cam0 (x,y,z) |               rect (x,y,z)
velo origin                      [0. 0. 0.] |     [-0.004 -0.076 -0.272] |     [-0.003 -0.075 -0.272]
10 m forward                  [10.  0.  0.] |        [0.071 0.072 9.727] |     [-0.     0.029  9.727]
10 m fwd, 5 m left            [10.  5.  0.] |     [-4.929  0.075  9.764] |     [-5.     0.082  9.728]
10 m fwd, 3 m up              [10.  0.  3.] |     [ 0.069 -2.928  9.771] |     [-0.032 -2.97   9.759]

확인: 전방→rect z 양수? True | 좌→rect x 음수? True | 상→rect y 음수? True


## 7. 수치 검증 — round-trip 과 역행렬

`rect_to_velo(velo_to_rect(p)) == p` 가 1e-6 m 안에서 성립해야 한다. 또 `invert_rigid(T) @ T == I` 를 확인한다.

In [ ]:
rng = np.random.default_rng(0)
pts = rng.uniform([-80, -40, -3], [80, 40, 3], size=(100_000, 3))
back = calib.rect_to_velo(calib.velo_to_rect(pts))
print("round-trip max |error| =", f"{np.abs(back - pts).max():.2e} m  (10만 점, |x|≤80 m)")
T = calib.T_velo_to_rect
print("|invert_rigid(T) @ T − I|max =", f"{np.abs(invert_rigid(T) @ T - np.eye(4)).max():.2e}")
print("|invert_rigid(T) − np.linalg.inv(T)|max =", f"{np.abs(invert_rigid(T) - np.linalg.inv(T)).max():.2e}")

round-trip max |error| = 8.53e-14 m  (10만 점, |x|≤80 m)
|invert_rigid(T) @ T − I|max = 1.11e-15
|invert_rigid(T) − np.linalg.inv(T)|max = 1.11e-15


## 8. 미리 보기 — 픽셀까지 가면 어디에 찍히나 (Phase 3 예고)

전방 10 m 의 점은 주점 $(c_x, c_y) = (609.6, 172.9)$ 근처에 맺혀야 한다. $t_x$ 항 때문에 $u$ 가 $t_x/d pprox 4.5$ px 만큼 오른쪽으로 옮겨진다.

In [ ]:
y = (to_homogeneous(test_pts) @ calib.P_velo_to_img.T)        # (4, 3) = [u·d, v·d, d]
for lab, row in zip(labels, y):
    d = row[2]
    if d > 0:
        print(f"{lab:20s} d = {d:6.3f} m → (u, v) = ({row[0]/d:7.1f}, {row[1]/d:7.1f}) px")
    else:
        print(f"{lab:20s} d = {d:6.3f} m → 카메라 뒤: 투영 불가")

velo origin          d = -0.269 m → 카메라 뒤: 투영 불가
10 m forward         d =  9.730 m → (u, v) = (  614.0,   175.0) px
10 m fwd, 5 m left   d =  9.731 m → (u, v) = (  243.2,   178.9) px
10 m fwd, 3 m up     d =  9.761 m → (u, v) = (  611.6,   -46.7) px


## 9. 정리

- **입력/출력**: 텍스트 2개 → `T_velo_to_cam (4×4)`, `R_rect_00 (4×4)`, `P_rect_02 (3×4)`, 합성 `P_velo_to_img (3×4)`.
- **좌표계**: velo (x 전방, y 좌, z 상) → cam0/rect (x 우, y 아래, z 전방) → img (u 우, v 아래).
- **핵심 수식**: $y = P_{rect,02} R_{rect,00} T_{velo\to cam} x$, 오른쪽부터 적용. 역변환은 $[R^T, -R^T t]$.
- **함정**: 곱 순서 · `R_rect_02` 오용 · `S_rect_02` 는 (W, H) · 텍스트 반올림으로 R 이 직교가 아님.
- **평가**: `tests/test_calibration.py` — 직교성, round-trip, 축 부호, 원점 위치.

### 자가 점검 질문
1. `P_rect_02 @ R_rect_00 @ T_velo_to_cam` 에서 Velodyne 점에 **가장 먼저** 적용되는 행렬은 무엇이며, 그 결과는 어느 프레임의 점인가?
2. Velodyne 원점을 rect 프레임으로 옮기면 z 가 음수인 이유는? 그 값은 어느 파일의 어느 숫자에서 오는가?
3. `np.linalg.inv` 대신 $R^T$ 로 역변환을 만들 수 있는 근거는 무엇이고, 그 전제가 깨졌을 때 어떤 증상이 나타났는가?